# 01 - Dataset Exploration

## UNSW-NB15 Dataset Analysis

**Phase 1: Dataset Understanding**

This notebook provides a comprehensive exploration of the UNSW-NB15 benchmark dataset,
including schema inspection, class distribution analysis, data quality checks, and
identification of potential leakage risks.

**Dataset Source:** https://research.unsw.edu.au/projects/unsw-nb15-dataset

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_loader import (
    load_raw_dataset,
    validate_dataset_schema,
    analyze_data_quality,
    analyze_class_distribution,
    check_train_test_overlap,
    identify_leakage_risks,
)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print(f'Python: {sys.version}')
print(f'Pandas: {pd.__version__}')
print(f'NumPy: {np.__version__}')

## 1. Load Raw Data

In [ ]:
# Load raw datasets without any modifications
df_train = load_raw_dataset('training')
df_test = load_raw_dataset('testing')

print(f'Training set: {df_train.shape[0]:,} rows x {df_train.shape[1]} columns')
print(f'Testing set:  {df_test.shape[0]:,} rows x {df_test.shape[1]} columns')
print(f'Total:        {df_train.shape[0] + df_test.shape[0]:,} records')

## 2. Schema Inspection

In [ ]:
# Display columns and their types
print('=== Column Names, Types, and Unique Counts ===')
for i, (col, dtype) in enumerate(df_train.dtypes.items()):
    nunique = df_train[col].nunique()
    print(f'{i:3d}. {col:25s} {str(dtype):12s}  nunique={nunique:>8,d}')

# Check if columns match between train and test
print(f'\nColumns match: {list(df_train.columns) == list(df_test.columns)}')

In [ ]:
# First 5 rows
df_train.head()

In [ ]:
# Statistical summary
df_train.describe()

## 3. Target Columns

In [ ]:
# Binary label distribution
print('=== Binary Label Distribution ===')
print('\nTraining set:')
print(df_train['label'].value_counts().sort_index())
print(f'Attack ratio: {df_train["label"].mean():.2%}')

print('\nTesting set:')
print(df_test['label'].value_counts().sort_index())
print(f'Attack ratio: {df_test["label"].mean():.2%}')

In [ ]:
# Multiclass attack_cat distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (name, df) in zip(axes, [('Training', df_train), ('Testing', df_test)]):
    counts = df['attack_cat'].value_counts().sort_values(ascending=True)
    colors = ['#2ecc71' if cat == 'Normal' else '#e74c3c' for cat in counts.index]
    counts.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(f'{name} Set - Attack Category Distribution')
    ax.set_xlabel('Count')
    
    # Add count labels
    for i, (val, cat) in enumerate(zip(counts.values, counts.index)):
        ax.text(val + 100, i, f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detailed class distribution comparison
train_dist = df_train['attack_cat'].value_counts()
test_dist = df_test['attack_cat'].value_counts()

comparison = pd.DataFrame({
    'Train Count': train_dist,
    'Train %': (train_dist / len(df_train) * 100).round(2),
    'Test Count': test_dist,
    'Test %': (test_dist / len(df_test) * 100).round(2),
})
comparison['Train/Test Ratio'] = (comparison['Train Count'] / comparison['Test Count']).round(2)
print(comparison.to_string())

## 4. Data Quality Analysis

In [ ]:
# Missing values
print('=== Missing Values ===')
print(f'Training: {df_train.isnull().sum().sum()} total missing cells')
print(f'Testing:  {df_test.isnull().sum().sum()} total missing cells')

# Show any columns with missing values
train_missing = df_train.isnull().sum()
if train_missing.any():
    print('\nColumns with missing values (training):')
    print(train_missing[train_missing > 0])
else:
    print('\nNo missing values in training set.')

test_missing = df_test.isnull().sum()
if test_missing.any():
    print('\nColumns with missing values (testing):')
    print(test_missing[test_missing > 0])
else:
    print('No missing values in testing set.')

In [ ]:
# Duplicate rows
print('=== Duplicate Rows ===')
train_dups = df_train.duplicated().sum()
test_dups = df_test.duplicated().sum()
print(f'Training: {train_dups:,} duplicates ({train_dups/len(df_train)*100:.2f}%)')
print(f'Testing:  {test_dups:,} duplicates ({test_dups/len(df_test)*100:.2f}%)')

In [ ]:
# Categorical feature analysis
print('=== Categorical Features ===')
for col in ['proto', 'service', 'state']:
    print(f'\n--- {col} ---')
    print(f'Training unique: {df_train[col].nunique()}')
    print(f'Testing unique:  {df_test[col].nunique()}')
    
    # Check for values in test not in train
    train_vals = set(df_train[col].unique())
    test_vals = set(df_test[col].unique())
    unseen = test_vals - train_vals
    if unseen:
        print(f'Values in test but NOT in train: {unseen}')
    else:
        print(f'All test values appear in training set.')

## 5. Leakage Risk Analysis

In [ ]:
# Correlation with binary label
numerical_train = df_train.select_dtypes(include=[np.number])
corrs = numerical_train.corrwith(df_train['label'].astype(float)).abs().sort_values(ascending=False)

print('=== Top 15 Correlations with Binary Label ===')
for col, corr in corrs.head(15).items():
    flag = ' *** LEAKAGE RISK' if corr > 0.9 and col != 'label' else ''
    print(f'  {col:25s}: {corr:.4f}{flag}')

print('\nNote: "id" column has high correlation (0.7272) with label.')
print('This is because IDs are assigned in order, and the dataset is sorted')
print('such that normal records come first. The "id" column MUST be excluded')
print('from features to prevent data leakage.')

In [ ]:
# Train/Test overlap check
overlap = check_train_test_overlap(df_train, df_test)
print(f'=== Train/Test Overlap ===')
print(f'Features compared: {overlap["feature_columns_compared"]}')
print(f'Overlapping records: {overlap["overlapping_records"]:,}')
print(f'Overlap as % of test set: {overlap["overlap_percentage_of_test"]:.2f}%')
print(f'\nNote: {overlap["overlap_percentage_of_test"]:.1f}% overlap between train and test.')
print('This is a known property of the official UNSW-NB15 split.')
print('We will use the official split as-is and document this overlap.')

## 6. Feature Distribution Analysis

In [ ]:
# Distribution of key features by attack category
key_features = ['dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sload', 'dload', 'rate']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for i, feat in enumerate(key_features):
    ax = axes[i]
    for cat in ['Normal', 'Generic', 'Exploits', 'DoS']:
        data = df_train[df_train['attack_cat'] == cat][feat]
        # Use log scale for skewed features
        data_plot = np.log1p(data.clip(lower=0))
        ax.hist(data_plot, bins=50, alpha=0.5, label=cat, density=True)
    ax.set_title(f'{feat} (log1p scale)')
    ax.legend(fontsize=7)

plt.suptitle('Feature Distributions by Attack Category (top 4 classes)', fontsize=14)
plt.tight_layout()
plt.savefig('../results/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap of numerical features
numerical_features = [c for c in df_train.select_dtypes(include=[np.number]).columns 
                      if c not in ['id', 'label']]

fig, ax = plt.subplots(figsize=(18, 14))
corr_matrix = df_train[numerical_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0, 
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.5},
            ax=ax, fmt='.1f')
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.savefig('../results/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Summary and Recommendations

### Key Findings

1. **Dataset Size**: 175,341 training + 82,332 testing = 257,673 total records
2. **Features**: 45 columns total (42 features + id + label + attack_cat)
3. **Data Types**: 3 categorical (proto, service, state), 39 numerical features, 2 targets, 1 ID
4. **Missing Values**: None
5. **Duplicates**: None
6. **Class Imbalance**: Severe (imbalance ratio ~431:1 between Generic and Worms)
7. **Train/Test Overlap**: ~10.4% of test records appear in training set

### Leakage Risks

- **`id` column**: Must be excluded (0.73 correlation with label due to ordering)
- **`label` and `attack_cat`**: These are target columns, not features

### Preprocessing Plan

1. Exclude: `id`, `label`, `attack_cat` from features
2. Encode categoricals: `proto`, `service`, `state` using LabelEncoder (fitted on train only)
3. Handle unseen categorical values in test set safely
4. Scale numerical features using StandardScaler (fitted on train only)
5. Consider class weighting for imbalanced categories (especially Worms, Shellcode, Backdoor)
6. For Track B (open-set), candidate withheld families: Worms (130 train), Shellcode (1133), Backdoor (1746)

In [ ]:
print('Phase 1 - Dataset Exploration Complete.')
print('\nNext step: Phase 2 - Multiclass Baseline (pending review)')